# Tool calling — and the description as prompt

**Session 6 · Track A · local Ollama**

Give the model a tool; measure how the description changes tool choice.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
# Local, free, offline. Needs a tool-capable model:  ollama pull llama3.2
import ollama


### Worked example

One typed tool, a good description vs a broken one, and tool-selection accuracy over prompts that should and should not trigger it.

_Runs on local Ollama. Needs a tool-capable model — `llama3.2`, `qwen2.5`, or `mistral-nemo`. Pull it first: `ollama pull llama3.2`. The base `llama3` does **not** do tool calling; this lab falls back to `llama3.2` if `OLLAMA_MODEL` is unset or `llama3`._


In [ ]:
# Worked example: the description IS the prompt - measure it
import os
import ollama

# Tool calling needs a tool-capable model; base `llama3` can't. Fall back to an
# installed llama3.2 tag (e.g. llama3.2:3b) if OLLAMA_MODEL is unset or `llama3`.
MODEL = os.environ.get("OLLAMA_MODEL", "llama3")
if MODEL in ("", "llama3", "llama3:latest"):
    installed = [m["model"] for m in ollama.list()["models"]]
    MODEL = next((m for m in installed if m.startswith("llama3.2")), "llama3.2")

def tool_spec(description):
    return [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    }]

GOOD   = ("Get the current weather for a city. Use whenever the user asks about "
          "weather, temperature, rain, or the forecast.")
BROKEN = "Does stuff."

SHOULD_CALL = ["What's the weather in Paris?", "Will it rain in Tokyo tomorrow?", "temperature in Cairo?"]
SHOULD_NOT  = ["Who painted the Mona Lisa?", "Translate 'hello' into Spanish."]

def selects_tool(prompt, description):
    r = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        tools=tool_spec(description),
        options={"temperature": 0.0},
    )
    return bool(r["message"].get("tool_calls"))

print(f"model: {MODEL}")
for name, desc in [("GOOD", GOOD), ("BROKEN", BROKEN)]:
    hits = sum(selects_tool(p, desc) for p in SHOULD_CALL)
    miss = sum(selects_tool(p, desc) for p in SHOULD_NOT)
    total = len(SHOULD_CALL) + len(SHOULD_NOT)
    print(f"{name:7} selection accuracy: {hits + (len(SHOULD_NOT) - miss)}/{total}  "
          f"(fired {hits}/{len(SHOULD_CALL)} wanted, {miss}/{len(SHOULD_NOT)} unwanted)")


## Your turn - vary the example

1. Add a second tool (e.g. `currency_convert`) and prompts that should route to each.
2. Degrade the GOOD description one word at a time - where does routing break?
3. Add an ambiguous prompt ("how's Paris?"). Which tool fires, and is that right?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
